# Beginner synthetic EHR workflow

This lesson uses only the repository's invented cohort. Run make research-demo first. The examples use aggregates and explain how the Bronze, Silver, and Gold layers relate.

In [ ]:
from src.core.config import load_config
from src.core.db import connect

config = load_config('configs/research_synthetic.duckdb.yaml')
con = connect(config)

## 1. Inspect the warehouse layers

Bronze retains source-like columns, Silver standardizes keys and timestamps, and Gold provides analysis-ready aggregates. Each table below is counted before any joins.

In [ ]:
con.execute('''
SELECT table_name
FROM information_schema.tables
WHERE table_name LIKE 'bronze_%' OR table_name LIKE 'silver_%' OR table_name LIKE 'gold_%'
ORDER BY table_name
''').fetchdf()

## 2. Keep the row grain explicit

Patients can have multiple admissions and an admission can have multiple events. Aggregate events before joining them to admissions.

In [ ]:
con.execute('''
WITH lab_counts AS (
  SELECT hadm_id, count(*) AS labs
  FROM silver_labs GROUP BY hadm_id
), admission_summary AS (
  SELECT admission_type, count(*) AS admissions, avg(coalesce(labs, 0)) AS mean_labs
  FROM silver_admissions a LEFT JOIN lab_counts l USING (hadm_id)
  GROUP BY admission_type
)
SELECT * FROM admission_summary ORDER BY admissions DESC
''').fetchdf()

## 3. Use a prediction time

The research model table uses first admission per patient. Predictors are available by the prediction time; LOS and mortality remain outcomes.

In [ ]:
con.execute('''
SELECT first_careunit, count(*) AS index_admissions,
       avg(hospital_los_days) AS mean_los_days,
       avg(in_hospital_mortality_flag) AS mortality_rate
FROM gold_research_index_admissions
WHERE prediction_time <= index_dischtime
GROUP BY first_careunit ORDER BY index_admissions DESC
''').fetchdf()

## 4. Read the reproducible artifacts

The research run writes aggregate feature definitions, repeated cross-validation, coefficients, and Shapley-style linear contribution summaries. These describe invented patterns only.

In [ ]:
from pathlib import Path
import pandas as pd

results = config.outputs_public_path / 'aggregate_results'
pd.read_csv(results / 'research_repeated_cv.csv').groupby('task').score.agg(['count', 'mean', 'std'])

Close the connection when finished with con.close(). Never treat the synthetic metrics or proxy features as clinical evidence.